# LeVJEPA feature visualization

This notebook loads the pretrained
[**LeVJEPA-VideoMix-Large**](https://huggingface.co/galilai-group/LeVJEPA-VideoMix-Large)
checkpoint (ViT-L/16, 303M parameters, trained self-supervised on 1.8M video clips)
and reproduces the two dense-feature visualizations from the paper:

1. **Patch-token PCA on an image** — the three leading principal components of the
   patch tokens, rendered as RGB. Although only the `[cls]` token is supervised during
   pretraining, the patch tokens organize by semantic region.
2. **Cosine similarity on a video** — the similarity between one query patch and every
   patch token of a 16-frame clip, shown as a heatmap per frame. High similarity stays
   confined to the object, showing the tokens are spatially precise as well.

Setup (from the repo root):

```bash
uv sync --extra notebook
uv run jupyter lab notebooks/feature_visualization.ipynb
```

The model runs on a CUDA GPU, an Apple Silicon Mac (MPS), or plain CPU — a CPU
forward pass takes a few seconds, a GPU makes it instant. The checkpoint download is
~1.2 GB on first use.

In [ ]:
import math
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.nn.functional as F
from PIL import Image
from transformers import AutoModel

if torch.cuda.is_available():
    device = "cuda"
elif torch.backends.mps.is_available():
    device = "mps"
else:
    device = "cpu"

model = AutoModel.from_pretrained(
    "galilai-group/LeVJEPA-VideoMix-Large", trust_remote_code=True
).eval().to(device)

# Geometry of the token grid: 16 frames (tubelet 1), 14x14 patches per frame at 224px.
NUM_FRAMES = 16
GRID = 14
IMG_SIZE = 224
EMBED_DIM = model.config.hidden_size if hasattr(model.config, "hidden_size") else 1024
print(f"loaded on {device}, {sum(p.numel() for p in model.parameters())/1e6:.1f}M params")

## Helpers

Preprocessing follows the model card: ImageNet normalization, 224px center crop.
The encoder returns `last_hidden_state` of shape `(B, 1 + 16*14*14, 1024)` — the
`[cls]` token followed by the patch tokens in `(frame, row, col)` order — which we
reshape into a `(16, 14, 14, 1024)` grid.

In [ ]:
IMAGENET_MEAN = torch.tensor([0.485, 0.456, 0.406])
IMAGENET_STD = torch.tensor([0.229, 0.224, 0.225])


def preprocess(img: Image.Image) -> torch.Tensor:
    """PIL image -> normalized (3, 224, 224) tensor (resize short side + center crop)."""
    w, h = img.size
    scale = IMG_SIZE / min(w, h)
    img = img.resize((round(w * scale), round(h * scale)), Image.BICUBIC)
    w, h = img.size
    left, top = (w - IMG_SIZE) // 2, (h - IMG_SIZE) // 2
    img = img.crop((left, top, left + IMG_SIZE, top + IMG_SIZE))
    x = torch.from_numpy(np.array(img.convert("RGB"))).float().div_(255)
    x = x.permute(2, 0, 1)
    return (x - IMAGENET_MEAN[:, None, None]) / IMAGENET_STD[:, None, None]


def denormalize(x: torch.Tensor) -> np.ndarray:
    """Normalized (3, H, W) tensor -> displayable HxWx3 array in [0, 1]."""
    x = x * IMAGENET_STD[:, None, None] + IMAGENET_MEAN[:, None, None]
    return x.clamp(0, 1).permute(1, 2, 0).numpy()


@torch.no_grad()
def encode(video: torch.Tensor) -> torch.Tensor:
    """(B, 3, 16, 224, 224) clip -> (B, 16, 14, 14, D) patch-token grid."""
    out = model(pixel_values=video.to(device))
    patch_tokens = out.last_hidden_state[:, 1:, :]  # drop [cls]
    return patch_tokens.reshape(-1, NUM_FRAMES, GRID, GRID, patch_tokens.shape[-1]).cpu()


def image_to_clip(img_t: torch.Tensor) -> torch.Tensor:
    """(3, 224, 224) image -> (1, 3, 16, 224, 224) clip by temporal repetition."""
    return img_t.unsqueeze(0).unsqueeze(2).repeat(1, 1, NUM_FRAMES, 1, 1)

## 1. Patch-token PCA on an image

A static image is repeated 16 times along the temporal axis (the protocol used for
ImageNet evaluation in the paper). We take the patch tokens of the **last** temporal
slot — under block-causal attention that's the frame with the most context — and
project them onto their three leading principal components, mapped to RGB.

Point `IMAGE_PATH` at your own image, or keep the bundled sample — a whippet on a
couch, the same kind of scene as the paper's PCA figure, where the dog should separate
cleanly from the furniture and the background.

In [ ]:
IMAGE_PATH = Path("data/dog.jpg")  # swap in your own image here

image = Image.open(IMAGE_PATH)
img_t = preprocess(image)
tokens = encode(image_to_clip(img_t))[0]  # (16, 14, 14, D)
feat = tokens[-1].reshape(GRID * GRID, -1)  # last temporal slot, (196, D)


def pca_rgb(feat: torch.Tensor, grid: int = GRID) -> np.ndarray:
    """(N, D) tokens -> (grid, grid, 3) RGB image of the 3 leading components."""
    feat = feat - feat.mean(dim=0)
    _, _, v = torch.pca_lowrank(feat, q=3)
    comps = (feat @ v).numpy()  # (N, 3)
    # Robust per-channel scaling to [0, 1] (2nd-98th percentile).
    lo, hi = np.percentile(comps, [2, 98], axis=0)
    comps = np.clip((comps - lo) / (hi - lo + 1e-8), 0, 1)
    return comps.reshape(grid, grid, 3)


rgb = pca_rgb(feat)

fig, axes = plt.subplots(1, 2, figsize=(8, 4))
axes[0].imshow(denormalize(img_t))
axes[0].set_title("input")
axes[1].imshow(rgb, interpolation="nearest")
axes[1].set_title("patch-token PCA (3 components as RGB)")
for ax in axes:
    ax.axis("off")
plt.tight_layout()
plt.show()

Tokens belonging to the same semantic region share a color, and the object separates
cleanly from the background — despite the patch tokens never receiving any loss during
pretraining. Try a few of your own images; scenes with a clear foreground object work
best at this 14x14 resolution.

## 2. Cosine similarity on a video

Now the real thing: a 16-frame clip. We pick one **query patch** — a (frame, row, col)
location on the token grid — and compute the cosine similarity between its token and
every patch token in the clip. If the representations are spatially precise, high
similarity should track the queried object across frames rather than smearing over
the scene.

Point `VIDEO_PATH` at any video file. If you've built the Walking Tours dataset
(`bash scripts/download_walking_tours.sh` from the repo root), the raw mp4s in
`data/walking_tours/videos/` work directly. Frames are sampled at ~7.5 fps to match
training, so 16 frames cover about two seconds.

In [ ]:
VIDEO_PATH = None   # e.g. "../data/walking_tours/videos/venice.mp4"
START_SEC = 0.0     # where in the video the clip starts
TARGET_FPS = 7.5    # training sampling rate


def load_clip(path, start_sec=0.0, target_fps=TARGET_FPS):
    """Video file -> normalized (1, 3, 16, 224, 224) clip. Uses decord if available
    (`--extra data`, Linux), PyAV otherwise (any platform)."""
    try:
        import decord

        vr = decord.VideoReader(str(path))
        native_fps = vr.get_avg_fps()
        step = max(1, round(native_fps / target_fps))
        start = round(start_sec * native_fps)
        idx = [min(start + i * step, len(vr) - 1) for i in range(NUM_FRAMES)]
        frames = vr.get_batch(idx).asnumpy()  # (16, H, W, 3) uint8
    except ImportError:
        import av

        with av.open(str(path)) as container:
            stream = container.streams.video[0]
            step = max(1, round(float(stream.average_rate) / target_fps))
            container.seek(int(start_sec * av.time_base))
            frames, i = [], 0
            for frame in container.decode(stream):
                if i % step == 0:
                    frames.append(frame.to_ndarray(format="rgb24"))
                if len(frames) >= NUM_FRAMES:
                    break
                i += 1
        while len(frames) < NUM_FRAMES:  # pad short tails with the last frame
            frames.append(frames[-1])
        frames = np.stack(frames)
    clip = torch.stack([preprocess(Image.fromarray(f)) for f in frames])  # (16, 3, H, W)
    return clip.permute(1, 0, 2, 3).unsqueeze(0)  # (1, 3, 16, 224, 224)


if VIDEO_PATH is None:
    candidates = sorted(Path("../data/walking_tours/videos").glob("*.mp4"))
    if candidates:
        VIDEO_PATH = candidates[0]
        print(f"using {VIDEO_PATH}")
    else:
        print(
            "No VIDEO_PATH set and no Walking Tours videos found - set VIDEO_PATH "
            "to any video file to run this section."
        )

if VIDEO_PATH is not None:
    clip = load_clip(VIDEO_PATH, START_SEC)
    video_tokens = encode(clip)[0]  # (16, 14, 14, D)

In [ ]:
# The query patch: frame index in [0, 15], row/col in [0, 13] on the 14x14 grid.
# Pick a patch sitting on an object in the frame shown below.
QUERY = (0, 7, 7)  # (frame, row, col) - center of the first frame by default

if VIDEO_PATH is not None:
    t_q, r_q, c_q = QUERY
    frame_img = denormalize(clip[0, :, t_q])
    fig, ax = plt.subplots(figsize=(4, 4))
    ax.imshow(frame_img)
    patch_px = IMG_SIZE // GRID
    ax.add_patch(
        plt.Rectangle(
            (c_q * patch_px, r_q * patch_px), patch_px, patch_px,
            edgecolor="red", facecolor="none", linewidth=2,
        )
    )
    ax.set_title(f"query patch: frame {t_q}, row {r_q}, col {c_q}")
    ax.axis("off")
    plt.show()

In [ ]:
if VIDEO_PATH is not None:
    tok = F.normalize(video_tokens.reshape(-1, video_tokens.shape[-1]), dim=-1)
    q = tok[t_q * GRID * GRID + r_q * GRID + c_q]
    sims = (tok @ q).reshape(NUM_FRAMES, GRID, GRID)  # (16, 14, 14), in [-1, 1]

    fig, axes = plt.subplots(4, 4, figsize=(12, 12))
    for t, ax in enumerate(axes.flat):
        ax.imshow(denormalize(clip[0, :, t]))
        hm = ax.imshow(
            sims[t], cmap="magma", alpha=0.6, vmin=0, vmax=1,
            extent=(0, IMG_SIZE, IMG_SIZE, 0), interpolation="bilinear",
        )
        if t == t_q:
            ax.add_patch(
                plt.Rectangle(
                    (c_q * patch_px, r_q * patch_px), patch_px, patch_px,
                    edgecolor="red", facecolor="none", linewidth=2,
                )
            )
        ax.set_title(f"frame {t}", fontsize=9)
        ax.axis("off")
    fig.colorbar(hm, ax=axes, shrink=0.6, label="cosine similarity to query")
    plt.show()

Bright regions are tokens similar to the query. Because the encoder is block-causal,
frames *before* the query frame never saw it — the similarity you see there comes purely
from the tokens representing the same content, not from attention between them.

Two things worth playing with:

- **Move the query** onto different objects (`QUERY` above) and watch the heatmap follow
  that object through the clip.
- **PCA also works on videos**: pass any `video_tokens[t].reshape(GRID * GRID, -1)` to
  `pca_rgb` from section 1 to see the per-frame semantic decomposition of a real clip.